# 問題
単語アナロジーの評価データをダウンロードし、国と首都に関する事例（: capital-common-countriesセクション）に対して、vec(2列目の単語) - vec(1列目の単語) + vec(3列目の単語)を計算し、そのベクトルと類似度が最も高い単語と、その類似度を求めよ。求めた単語と類似度は、各事例と一緒に記録せよ。

In [2]:
from gensim.models import KeyedVectors
import csv
from pathlib import Path

# ===== 1) データ読み込み（capital-common-countries だけ抽出、4列目は捨てる） =====
def load_capital_common_countries(txt_path: str):
    items = []
    recording = False
    with open(txt_path, encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(":"):
                # セクション切り替え
                recording = (line == ": capital-common-countries")
                continue
            if recording:
                parts = line.split()
                if len(parts) >= 3:
                    # 4列目（正解ラベル）は今回の要件では使わないので切り捨て
                    items.append(parts[:3])  # [a, b, c]
    return items  # [["Athens","Greece","Baghdad"], ...]


# ===== 2) アナロジー計算（vec(b) - vec(a) + vec(c)）→ 最類似語＆類似度 =====
def run_analogy(items, kv: KeyedVectors):
    rows = []
    for a, b, c in items:
        # 3語すべてが語彙にあるかチェック
        if (a not in kv) or (b not in kv) or (c not in kv):
            rows.append({
                "a": a, "b": b, "c": c,
                "pred_word": "", "similarity": "",
                "note": f"OOV: {'/'.join([w for w in (a,b,c) if w not in kv])}"
            })
            continue

        # most_similar を使う（vec(b)+vec(c)-vec(a) と等価）
        pred_word, sim = kv.most_similar(positive=[b, c], negative=[a], topn=1)[0]

        rows.append({
            "a": a, "b": b, "c": c,
            "pred_word": pred_word,
            "similarity": f"{sim:.4f}",
            "note": ""
        })

    return rows


# ===== 3) 実行 =====
# アナロジーデータ
txt_path = "単語ベクトルの質問.txt"  

# GoogleNews Word2Vec（300次元）
kv_path = "/Users/nakamuratuzumi/Jupyterlab/言語処理100本ノック 2025/第06章_単語ベクトル/GoogleNews-vectors-negative300.bin"

# セクション抽出
items = load_capital_common_countries(txt_path)

# モデル読み込み（最初に1回だけ）
kv = KeyedVectors.load_word2vec_format(kv_path, binary=True)

# 実行 & 保存
results = run_analogy(items, kv)

# 先頭5件だけ表示
for r in results[:5]:
    print(r)


{'a': 'Athens', 'b': 'Greece', 'c': 'Baghdad', 'pred_word': 'Iraqi', 'similarity': '0.6352', 'note': ''}
{'a': 'Athens', 'b': 'Greece', 'c': 'Bangkok', 'pred_word': 'Thailand', 'similarity': '0.7138', 'note': ''}
{'a': 'Athens', 'b': 'Greece', 'c': 'Beijing', 'pred_word': 'China', 'similarity': '0.7236', 'note': ''}
{'a': 'Athens', 'b': 'Greece', 'c': 'Berlin', 'pred_word': 'Germany', 'similarity': '0.6735', 'note': ''}
{'a': 'Athens', 'b': 'Greece', 'c': 'Bern', 'pred_word': 'Switzerland', 'similarity': '0.4920', 'note': ''}


In [3]:
import pandas as pd

df = pd.DataFrame(results)
df.head(10)

,a,b,c,pred_word,similarity,note
0,Athens,Greece,Baghdad,Iraqi,0.6352,
1,Athens,Greece,Bangkok,Thailand,0.7138,
2,Athens,Greece,Beijing,China,0.7236,
3,Athens,Greece,Berlin,Germany,0.6735,
4,Athens,Greece,Bern,Switzerland,0.4920,
5,Athens,Greece,Cairo,Egypt,0.7528,
6,Athens,Greece,Canberra,Australia,0.5837,
7,Athens,Greece,Hanoi,Viet_Nam,0.6276,
8,Athens,Greece,Havana,Cuba,0.6461,
9,Athens,Greece,Helsinki,Finland,0.6900,


In [4]:
# note列が空でないものを抽出
df_with_note = df[df["note"] != ""]

# または NaN も考慮する場合
df_with_note = df[df["note"].notna() & (df["note"] != "")]

df_with_note.head()

,a,b,c,pred_word,similarity,note
